# Tugas Terstruktur 2

## Diagnosis Bias-Variance dan Regularisasi

**Mata Kuliah** Data Science (TI24425) &middot; 3 sks (Teori)
**Program Studi** Teknologi Informasi &middot; Politeknik Negeri Madiun
**Semester** Genap &middot; Tahun Akademik 2026/2027
**Cakupan materi** Pertemuan 4 dan 5
**Bobot** 4% dari nilai akhir
**Bentuk** Kerja kelompok, tiga orang &middot; diberikan pada pertemuan 5

**Materi rujukan** Pertemuan 4 dan 5 &middot; **Sub-CPMK 5**

---

### Identitas Kelompok

| | Nama Lengkap | NPM | Peran | Bagian |
|---|---|---|---|---|
| 1 | | | Diagnosa Kurva | Bagian A |
| 2 | | | Penelusur Kompleksitas | Bagian B |
| 3 | | | Penyetel Regularisasi | Bagian C |

| | |
|---|---|
| **Kelas** | |
| **Judul dataset** | |
| **Tanggal pengumpulan** | |


---

## Petunjuk Pengerjaan

### Kesinambungan dengan tugas sebelumnya

Tugas ini memakai **dataset yang sama** dengan Tugas Terstruktur 1. Jangan berganti dataset. Seluruh rangkaian tugas terstruktur dirancang menumpuk, dan hasilnya menjadi bekal langsung bagi Studi Kasus Akhir pada pertemuan 14 dan 15.

### Cara mengerjakan

- Setiap anggota mengerjakan **satu bagian** sesuai perannya. **Bagian D dikerjakan bersama.**
- Sel bertanda `[KODE]` diisi kode Python. Sel bertanda `[URAIAN]` diisi tulisan Anda sendiri.
- **Kode yang berjalan tanpa uraian tidak memperoleh nilai.** Yang dinilai adalah penalaran.
- Jangan menghapus sel pertanyaan. Tulis jawaban tepat di bawahnya.

### Status kode pada mata kuliah teori

Mata kuliah ini adalah mata kuliah teori. Kegiatan berbasis komputer berlangsung di luar jam tatap muka sebagai **Penugasan Terstruktur**. Kode yang diminta sengaja dibuat sederhana dan dapat dikerjakan dengan menyesuaikan nama kolom. Yang dinilai tetap kualitas analisis.

### Menyimpan sebagai PDF

1. Jalankan seluruh sel: **Run &rarr; Run All Cells**. Pastikan tidak ada pesan galat.
2. **File &rarr; Print** (`Ctrl` + `P`), pilih tujuan **Save as PDF**.
3. Beri nama `TT2_<Kelas>_<NamaKelompok>.pdf`, unggah bersama berkas `.ipynb` ke LMS.

### Penggunaan AI generatif

Diperbolehkan sebagai alat bantu, dengan syarat dicantumkan pada Lampiran di akhir berkas: bagian mana yang dibantu dan bagaimana hasilnya Anda verifikasi.


---

## Persiapan

In [ ]:
# [KODE] Persiapan pustaka — jalankan apa adanya
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', 50); pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7.5, 4); plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
np.random.seed(42)
print('Pustaka siap.')

In [ ]:
# [KODE] Pemuatan dan penyiapan data — sesuaikan tiga baris pertama saja
NAMA_BERKAS  = 'nama_berkas_dataset.csv'
PEMISAH      = ','
KOLOM_TARGET = 'ganti_dengan_nama_kolom_target'

df = pd.read_csv(NAMA_BERKAS, sep=PEMISAH)

def siapkan(df, target, maks_kategori=15):
    """Menyiapkan X dan y: buang baris tanpa target, encoding kategorik, isi nilai hilang numerik."""
    d = df.dropna(subset=[target]).copy()
    y = d[target]
    X = d.drop(columns=[target])
    # buang kolom kategorik dengan terlalu banyak nilai unik (kemungkinan penanda identitas)
    buang = [c for c in X.columns
             if X[c].dtype == 'object' and X[c].nunique() > maks_kategori]
    if buang:
        print('Kolom dikeluarkan karena terlalu banyak nilai unik:', buang)
        X = X.drop(columns=buang)
    X = pd.get_dummies(X, drop_first=True)              # one-hot untuk kategorik
    X = X.fillna(X.median(numeric_only=True))            # isi nilai hilang numerik
    return X, y

X, y = siapkan(df, KOLOM_TARGET)
print('Ukuran X :', X.shape)
print('Ukuran y :', y.shape)
print('Tipe target:', 'kategorik / klasifikasi' if y.nunique() <= 10 else 'numerik / regresi')

---
---

# Bagian A &mdash; Diagnosis Kurva Pembelajaran

**Dikerjakan oleh Anggota 1 &middot; Diagnosa Kurva**

Bagian ini **tidak memakai dataset kelompok Anda**. Data pada bagian ini disediakan dosen agar seluruh kelompok mendiagnosis kasus yang sama.

## A.1 Empat Model, Empat Kondisi

Empat model dilatih pada masalah yang sama. Berikut galat akhir masing-masing.

| Model | Galat data latih | Galat data validasi | Kondisi? | Satu tindakan perbaikan |
|---|---|---|---|---|
| M1 | 0,02 | 4,80 | | |
| M2 | 3,90 | 4,05 | | |
| M3 | 0,45 | 0,61 | | |
| M4 | 0,10 | 0,15 | | |

**[URAIAN A.1]** Isi dua kolom terakhir pada tabel di atas, lalu jawab:

1. Untuk **setiap** model, sebutkan kondisinya: underfitting, overfitting, seimbang, atau mencurigakan. Jelaskan dasar penilaian Anda — bukan hanya menyebut istilahnya.
2. Model M4 memiliki galat paling kecil di kedua kolom. Apakah otomatis berarti M4 yang terbaik? Apa **satu informasi tambahan** yang wajib Anda tanyakan sebelum memutuskan?

> _Tulis jawaban Anda di sini._


## A.2 Membaca Kurva Pembelajaran

Sel berikut menggambar tiga kurva pembelajaran yang disediakan dosen. Ketiganya berasal dari tiga model berbeda pada masalah yang sama.

In [ ]:
# [KODE] Kurva pembelajaran yang disediakan dosen — jalankan apa adanya
epoch = np.arange(0, 100, 10)

kurva = {
    'Model P': {'latih': [1.10,0.72,0.51,0.38,0.29,0.22,0.17,0.13,0.10,0.07],
                'validasi': [1.12,0.78,0.60,0.50,0.46,0.47,0.52,0.60,0.70,0.82]},
    'Model Q': {'latih': [1.15,0.98,0.90,0.86,0.84,0.83,0.82,0.82,0.81,0.81],
                'validasi': [1.18,1.02,0.94,0.90,0.88,0.87,0.86,0.86,0.85,0.85]},
    'Model R': {'latih': [1.12,0.70,0.48,0.36,0.28,0.23,0.20,0.18,0.17,0.16],
                'validasi': [1.14,0.75,0.54,0.42,0.34,0.30,0.28,0.27,0.27,0.26]},
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, (nama, k) in zip(axes, kurva.items()):
    ax.plot(epoch, k['latih'], marker='o', ms=3, label='data latih')
    ax.plot(epoch, k['validasi'], marker='s', ms=3, label='data validasi')
    ax.set_title(nama); ax.set_xlabel('Epoch')
axes[0].set_ylabel('Galat'); axes[0].legend()
plt.tight_layout(); plt.show()

**[URAIAN A.2]** Untuk **setiap** kurva, jawab keempat pertanyaan berikut.

| | Model P | Model Q | Model R |
|---|---|---|---|
| Kondisi yang terbaca | | | |
| Bukti dari bentuk kurvanya | | | |
| Pada epoch berapa sebaiknya pelatihan dihentikan? | | | |
| Satu tindakan perbaikan yang tepat | | | |

Kemudian jawab: teknik apa yang seharusnya diterapkan pada Model P agar kerugian akibat kondisinya tidak terjadi, dan mengapa teknik itu **menuntut adanya data validasi**?

> _Tulis jawaban Anda di sini._


---
---

# Bagian B &mdash; Kurva Kompleksitas pada Dataset Kelompok

**Dikerjakan oleh Anggota 2 &middot; Penelusur Kompleksitas**

Bagian ini memakai dataset kelompok Anda. Tujuannya menunjukkan bias-variance trade-off secara empiris.

In [ ]:
# [KODE] Membangun kurva kompleksitas dengan menaikkan kedalaman pohon
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

KLASIFIKASI = y.nunique() <= 10          # ditentukan otomatis; ubah manual bila perlu

X_latih, X_val, y_latih, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=y if KLASIFIKASI else None)

kedalaman = range(1, 16)
galat_latih, galat_val = [], []

for d in kedalaman:
    if KLASIFIKASI:
        m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_latih, y_latih)
        galat_latih.append(1 - accuracy_score(y_latih, m.predict(X_latih)))
        galat_val.append(1 - accuracy_score(y_val, m.predict(X_val)))
    else:
        m = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_latih, y_latih)
        galat_latih.append(mean_squared_error(y_latih, m.predict(X_latih)))
        galat_val.append(mean_squared_error(y_val, m.predict(X_val)))

hasil = pd.DataFrame({'kedalaman': list(kedalaman),
                      'galat_latih': np.round(galat_latih, 4),
                      'galat_validasi': np.round(galat_val, 4)})
print(hasil.to_string(index=False))

In [ ]:
# [KODE] Menggambar kurva kompleksitas
fig, ax = plt.subplots()
ax.plot(hasil['kedalaman'], hasil['galat_latih'], marker='o', ms=4, label='data latih')
ax.plot(hasil['kedalaman'], hasil['galat_validasi'], marker='s', ms=4, label='data validasi')
titik = hasil.loc[hasil['galat_validasi'].idxmin()]
ax.axvline(titik['kedalaman'], ls='--', lw=1, color='gray')
ax.set_xlabel('Kedalaman maksimum pohon'); ax.set_ylabel('Galat')
ax.set_title('Kurva kompleksitas pada dataset kelompok'); ax.legend()
plt.tight_layout(); plt.show()

print('Kedalaman dengan galat validasi terkecil :', int(titik['kedalaman']))
print('Galat validasi terkecil                  :', round(titik['galat_validasi'], 4))

**[URAIAN B.1]** Jawab kelima pertanyaan berikut.

1. Pada kedalaman berapa galat validasi mencapai titik terendah? Apa yang terjadi **sesudah** titik itu, dan mengapa?
2. Bagaimana perilaku galat **data latih** seiring bertambahnya kedalaman? Jelaskan mengapa demikian.
3. Pada kedalaman berapa model Anda mulai **overfitting**? Sebutkan bukti angkanya.
4. Apakah ada bagian kurva yang menunjukkan **underfitting**? Bila ada, di rentang kedalaman berapa?
5. Kaitkan temuan ini dengan istilah **bias** dan **variance**: bagian mana dari kurva yang didominasi bias tinggi, dan bagian mana yang didominasi variance tinggi?

> _Tulis jawaban Anda di sini._


---
---

# Bagian C &mdash; Regularisasi dan Pemilihan λ

**Dikerjakan oleh Anggota 3 &middot; Penyetel Regularisasi**

Bagian ini menelusuri pengaruh kekuatan regularisasi terhadap koefisien model linier.

In [ ]:
# [KODE] Jalur koefisien Ridge dan Lasso terhadap perubahan lambda
from sklearn.linear_model import Ridge, Lasso, LogisticRegression

# Hanya kolom numerik, diskalakan lebih dulu — penskalaan wajib sebelum regularisasi
X_num = X.select_dtypes(include=[np.number])
skala = StandardScaler().fit(X_num)
Xs = pd.DataFrame(skala.transform(X_num), columns=X_num.columns)

y_num = pd.factorize(y)[0] if KLASIFIKASI else y.values

daftar_lambda = [0.01, 0.1, 1, 10, 100]
jalur_ridge, jalur_lasso = {}, {}

for lam in daftar_lambda:
    jalur_ridge[lam] = Ridge(alpha=lam).fit(Xs, y_num).coef_
    jalur_lasso[lam] = Lasso(alpha=lam, max_iter=5000).fit(Xs, y_num).coef_

koef_ridge = pd.DataFrame(jalur_ridge, index=Xs.columns).round(3)
koef_lasso = pd.DataFrame(jalur_lasso, index=Xs.columns).round(3)

print('KOEFISIEN RIDGE (L2) pada berbagai lambda'); print(koef_ridge.head(10)); print()
print('KOEFISIEN LASSO (L1) pada berbagai lambda'); print(koef_lasso.head(10))

In [ ]:
# [KODE] Berapa banyak koefisien yang menjadi tepat nol
ringkas = pd.DataFrame({
    'lambda'            : daftar_lambda,
    'ridge_koef_nol'    : [(np.abs(koef_ridge[l]) < 1e-8).sum() for l in daftar_lambda],
    'lasso_koef_nol'    : [(np.abs(koef_lasso[l]) < 1e-8).sum() for l in daftar_lambda],
    'ridge_jumlah_besar': [np.abs(koef_ridge[l]).sum().round(3) for l in daftar_lambda],
    'lasso_jumlah_besar': [np.abs(koef_lasso[l]).sum().round(3) for l in daftar_lambda],
})
print('Jumlah kolom fitur:', Xs.shape[1])
print(ringkas.to_string(index=False))

**[URAIAN C.1]** Jawab kelima pertanyaan berikut berdasarkan kedua keluaran di atas.

1. Bagaimana perilaku koefisien Ridge seiring naiknya λ? Apakah ada yang menjadi **tepat nol**?
2. Bagaimana perilaku koefisien Lasso? Berapa banyak yang menjadi tepat nol pada λ terbesar?
3. Jelaskan **mengapa** keduanya berperilaku berbeda, dengan merujuk bentuk penaltinya masing-masing.
4. Fitur mana yang **paling bertahan** dari pengecilan pada Lasso? Apakah bertahannya fitur itu masuk akal menurut pemahaman Anda terhadap dataset ini?
5. Mengapa penskalaan **wajib** dilakukan sebelum regularisasi? Apa yang akan terjadi bila langkah itu dilewati?

> _Tulis jawaban Anda di sini._


In [ ]:
# [KODE] Memilih lambda dengan cross-validation lima lipatan
from sklearn.model_selection import cross_val_score

skor = []
for lam in [0.001, 0.01, 0.1, 1, 10, 100]:
    pipa = Pipeline([('skala', StandardScaler()), ('model', Ridge(alpha=lam))])
    s = cross_val_score(pipa, X_num, y_num, cv=KFold(5, shuffle=True, random_state=42),
                        scoring='neg_mean_squared_error')
    skor.append({'lambda': lam, 'mse_rerata': round(-s.mean(), 4), 'simpangan': round(s.std(), 4)})

tabel_lam = pd.DataFrame(skor)
print(tabel_lam.to_string(index=False))
print()
print('Lambda terbaik :', tabel_lam.loc[tabel_lam['mse_rerata'].idxmin(), 'lambda'])

**[URAIAN C.2]** Jawab tiga pertanyaan berikut.

1. Nilai λ mana yang terpilih, dan berapa selisih galatnya dibanding λ terburuk?
2. Perhatikan kolom **simpangan**. Adakah nilai λ yang galatnya rendah tetapi simpangannya besar? Apa artinya, dan mengapa hal itu perlu diwaspadai?
3. Perhatikan bahwa penskalaan dijalankan **di dalam** `Pipeline`, bukan sebelum cross-validation. Jelaskan mengapa urutan itu penting, dan apa yang bocor seandainya penskalaan dilakukan lebih dulu pada seluruh data.

> _Tulis jawaban Anda di sini._


---
---

# Bagian D &mdash; Sintesis dan Rekomendasi

**Dikerjakan bersama oleh ketiga anggota.**

## D.1 Diagnosis Kelompok

**[URAIAN D.1]** Berdasarkan seluruh temuan Bagian A sampai C, isi tabel berikut.

| Butir | Isian |
|---|---|
| Kondisi model kelompok kami saat ini (underfitting / overfitting / seimbang) | |
| Bukti angka yang mendasarinya | |
| Sumbernya lebih dominan bias atau variance? | |
| Dasar penentuan tersebut | |


## D.2 Rekomendasi Perbaikan

**[URAIAN D.2]** Susun **tiga** rekomendasi perbaikan, diurutkan dari yang paling mendesak. Setiap rekomendasi wajib disertai alasan dan perkiraan akibatnya.

| Urutan | Tindakan yang direkomendasikan | Alasan | Yang diharapkan berubah |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

Kemudian jawab: adakah rekomendasi yang **menurunkan** kinerja pada data latih tetapi tetap Anda usulkan? Bila ada, jelaskan mengapa itu justru tanda yang baik.

> _Tulis jawaban Anda di sini._


## D.3 Batas yang Tidak Dapat Ditembus

**[URAIAN D.3]** Jawab dua pertanyaan berikut dengan jujur.

1. Menurut Anda, berapa kira-kira **galat tak tereduksi** pada masalah ini, dan dari mana sumbernya? Sebutkan faktor nyata yang tidak terekam dalam dataset Anda.
2. Seandainya kelompok Anda berhasil mencapai galat nol pada data latih, apa yang justru perlu Anda curigai?

> _Tulis jawaban Anda di sini._


## D.4 Pembagian Kerja

| Anggota | Bagian | Perkiraan waktu | Kesulitan terbesar |
|---|---|---|---|
| 1 | Bagian A | | |
| 2 | Bagian B | | |
| 3 | Bagian C | | |
| Bersama | Bagian D | | |


---

# Lampiran &mdash; Pernyataan Penggunaan AI Generatif

| Bagian yang dibantu | Nama alat | Bentuk bantuan | Cara kami memverifikasi |
|---|---|---|---|
| | | | |
| | | | |

Dengan ini kami menyatakan bahwa seluruh analisis, justifikasi, dan kesimpulan dalam berkas ini merupakan hasil penalaran kelompok kami sendiri, dan seluruh bantuan alat AI generatif telah kami cantumkan secara jujur.

| Anggota 1 | Anggota 2 | Anggota 3 |
|---|---|---|
| ( ......................... ) | ( ......................... ) | ( ......................... ) |


---

# Rubrik Penilaian

Total 100 poin, dikonversi menjadi bobot 4% pada nilai akhir.

| Bagian | Kriteria | Poin |
|---|---|---|
| A | Ketepatan mendiagnosis empat kondisi model dan menjawab pertanyaan jebakan M4 | 12 |
| A | Ketepatan membaca ketiga kurva pembelajaran beserta titik hentinya | 13 |
| B | Ketepatan menafsirkan kurva kompleksitas dan menunjuk titik overfitting | 15 |
| B | Ketepatan mengaitkan temuan dengan istilah bias dan variance | 10 |
| C | Ketepatan membedakan perilaku koefisien Ridge dan Lasso beserta alasannya | 15 |
| C | Pemahaman mengenai kewajiban penskalaan dan letak Pipeline dalam cross-validation | 10 |
| D | Ketepatan diagnosis kelompok | 10 |
| D | Kelayakan tiga rekomendasi dan kekuatan alasannya | 15 |
| | **Jumlah** | **100** |

### Ketentuan penilaian

- **Kode yang berjalan tanpa uraian bernilai nol** untuk butir yang bersangkutan.
- Jawaban umum yang dapat dipakai untuk dataset mana pun **tidak memperoleh nilai penuh**.
- Menyebut suatu hal keliru **tanpa menjelaskan mengapa** hanya memperoleh separuh poin.
- Mengakui keterbatasan secara jujur **menambah** nilai; menutupinya mengurangi nilai.
- Keterlambatan dikenakan pengurangan 10% nilai per hari kerja, maksimal tiga hari kerja.

### Pembobotan khusus tugas ini

Sesuai kriteria yang disampaikan pada pertemuan 5: **ketepatan diagnosis 40%**, **kelayakan rekomendasi 30%**, dan **kekuatan alasan 30%**. Rekomendasi tanpa alasan tidak memperoleh nilai penuh, sekalipun tindakannya benar.